In [5]:
import pandas as pd
import joblib
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
# Load dataset
csv_path = "/content/cattle_feed_data_1000.csv" # Ensure this is in the same directory or give full path
data = pd.read_csv(csv_path)

# Define input and output features
X = data[['Breed', 'Weight_kg', 'Age_years', 'Purpose', 'Weather']]
y = data[['Dry_Fodder_kg', 'Concentrate_kg', 'Green_Fodder_kg']]

# Define categorical columns
categorical_features = ['Breed', 'Purpose', 'Weather']

# Preprocessing: OneHotEncode categorical features
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)],
    remainder='passthrough'
)

# Try importing LightGBM with GPU
try:
    import lightgbm as lgb
    print("✅ Using LightGBM with GPU")
    base_model = lgb.LGBMRegressor(device='gpu', n_estimators=200, random_state=42)
except ImportError:
    from sklearn.ensemble import RandomForestRegressor
    print("⚠️ LightGBM not found. Falling back to RandomForestRegressor (CPU)")
    base_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Wrap LightGBM in MultiOutputRegressor
multi_output_model = MultiOutputRegressor(base_model)

# Create full pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', multi_output_model)
])

# Train the model
print("🚀 Training started...")
pipeline.fit(X, y)
print("✅ Training completed.")

# Save the model
model_path = "cattle_diet_predictor.pkl"
joblib.dump(pipeline, model_path)
print(f"💾 Model saved to {model_path}")


✅ Using LightGBM with GPU
🚀 Training started...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 280
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 12
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 5 dense feature groups (0.01 MB) transferred to GPU in 0.000480 secs. 0 sparse feature groups
[LightGBM] [Info] Start training from score 5.038100
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 280
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 12
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin en

In [6]:
import pandas as pd
import joblib

# Load trained model
model = joblib.load("cattle_diet_predictor.pkl")

# Predict for new input
new_input = pd.DataFrame([{
    'Breed': 'Jersey',
    'Weight_kg': 520,
    'Age_years': 6,
    'Purpose': 'Meat',
    'Weather': 'Hot'
}])

predicted_diet = model.predict(new_input)
print("Predicted [Dry_Fodder, Concentrate, Green_Fodder_kg]:", predicted_diet[0])


Predicted [Dry_Fodder, Concentrate, Green_Fodder_kg]: [ 6.46741747  2.29859503 10.35280895]
